In [ ]:
# ==============================================================================
# CELL 1 / PHASE 1: GETTY ART & ARCHITECTURE THESAURUS (AAT) INGESTION
# 
# ARCHITECTURE NOTE: AAT is a massive, heavily polyhierarchical ontology. 
# A single concept (e.g., "choirstalls") can have multiple valid broader 
# parents (e.g., "church furniture" and "seating furniture"). 
#
# SSSOM ALIGNMENT STRATEGY: 
# To manage polyhierarchy in a flat CSV, we use a "Hybrid Approach". We query 
# Getty's custom `gvp:parentString` to extract a single, linear preferred 
# path for the human-readable `Hierarchy_Path` column. Simultaneously, we 
# extract all immediate parent IDs via `skos:broader` and pipe-separate them 
# in the `Parent_IDs` column to preserve the true mathematical graph. 
# To optimize speed, we use `gvp:broaderExtended` to grab entire subtrees in 
# a single SPARQL query rather than crawling node-by-node.
# ==============================================================================

import os
import sys
import requests
import pandas as pd
import time
from dotenv import load_dotenv

# --- 1. Environment & Central Schema Setup ---
load_dotenv("../../config/.env") 
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))

sys.path.append(config_dir)
try:
    from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER
except ImportError:
    raise ImportError("CRITICAL: Could not find ingest_schema_manager.py. Check PYTHONPATH.")

# --- 2. Registry Lookup ---
SOURCE_PREFIX = "AAT"
registry_data = get_registry_info(
    prefix=SOURCE_PREFIX, 
    config_dir=config_dir,
    fallback_name="Getty Art & Architecture Thesaurus",
    fallback_uri="http://vocab.getty.edu/aat/"
)
SOURCE_NAME = registry_data['Source_Name']
raw_ingest_file = os.path.join(raw_data_dir, f"raw_{SOURCE_PREFIX.lower()}.csv")

GETTY_SPARQL_URL = "http://vocab.getty.edu/sparql.json"

# --- 3. Seed List ---
AAT_SEEDS = {
    # --- Original High-Level Seeds ---
    "religious organizations": "300250844",
    "people in religion and related occupations": "300025746",
    "religions and religious concepts": "300055980",
    "religious structures": "300120364",
    "religious centers": "300387272",
    "religious events": "300069688",
    "religious institutions": "300343472",

    # --- New Seeds from Lateral Discovery ---
    "<religion and related disciplines>": "300054299",
    "occult sciences": "300054587",
    "christenings": "300137692",
    "religious holidays": "300069349",
    "religious seasons": "300192156",
    "rituals (events)": "300065284",
    "pilgrimages": "300069279",
    "conclaves": "300404871",
    "yoga (activity)": "300264366",
    "ritual destruction (process)": "300458782",
    "religious divisions": "300236153",
    "papacies": "300379037",
    "religious characters": "300412107",
    "retablistas": "300311286",
    "<people in the occult sciences>": "300239172",
    "buddhas (people)": "300404698",
    "pilgrims (people)": "300266259",
    "saints": "300150555",
    "teixiptlahuan": "300404940",
    "cults (group or movement)": "300251847",
    "beati (people)": "300150482",
    "gnostics": "300236781",
    "holy water": "300264702",
    "chrism": "300265211",
    "peyote (plant material)": "300249790",
    "missions (complexes)": "300410262",
    "religious complexes": "300263741",
    "sacred sites": "300386998",
    "archiepiscopal sees": "300387236",
    "episcopal sees": "300387235",
    "agrahāra": "300386139",
    "altar tombs": "300005928",
    "sacred tanks": "300458692",
    "cross monuments": "300133797",
    "clergy houses": "300005639",
    "abbot's quarters": "300451314",
    "ceremonial containers": "300197576",
    "christening cups": "300391137",
    "liturgical costume": "300210414",
    "religious habits": "300390957",
    "personal shrines": "300391181",
    "religious visual works": "300234093",
    "totems": "300255468",
    "shamanic art": "300457672",

    # --- Additional Seeds from Phase 2 Lateral Harvest ---
    "rites": "300312160",
    "fasts (events)": "300069095",
    "ritual rebuilding (process)": "300458781",
    "convents (groups)": "300312089",
    "religious schools (institutions)": "300343470",
    "seminarians": "300404369",
    "altar makers": "300420020",
    "altarpiece artists": "300435119",
    "religious poetry": "300417816",
    "prayers (literary genre)": "300026452",
    "religious literature (genre)": "300417817",
    "alms (gifts)": "300400538",
    "religious schools (buildings)": "300448956",
    "prayer stools (stools, general)": "300038454",
    "baptismal candles": "300404224",
    "epiphany candles": "300404229",
    "paschal candles": "300190806",
    "epiphany candlesticks": "300404246",
    "religious objects": "300234098",
    "religious calendars": "300026754",
    "Torah scrolls": "300419304",
    "sermons": "300026669",
    "totem poles": "300184947",

    # --- Further Additions from Lateral Harvest ---
    "parochial schools (buildings)": "300006580",
    "Sunday schools (buildings)": "300132249",
    "madrasas (buildings)": "300006563",
    "yeshivas (buildings)": "300048798",
    "sacred objects": "300234190",
    "prayers (compositions)": "300400521"
}

# --- 4. Persistent Progress & Schema Drift Detection ---
if os.path.exists(raw_ingest_file):
    existing_df = pd.read_csv(raw_ingest_file)
    if list(existing_df.columns) != COLUMN_ORDER:
        print(f"[!] Schema mismatch detected. Deleting outdated {os.path.basename(raw_ingest_file)}...")
        os.remove(raw_ingest_file)
        visited_uris = set()
    else:
        visited_uris = set(existing_df['URI'].unique().tolist())
        print(f"[*] Resuming: Found {len(visited_uris)} concepts already in {raw_ingest_file}")
else:
    visited_uris = set()
    print("[*] Starting fresh Getty AAT ingest.")

# --- 5. Execution Loop ---
print(f"\nStarting Getty AAT ingest for {len(AAT_SEEDS)} branches...")

for seed_name, seed_id in AAT_SEEDS.items():
    print(f"\n[*] Processing branch: {seed_name} (aat:{seed_id})...")
    
    # We grab the descendants AND the seed itself, plus preferred path and crosswalks
    QUERY = f"""
    SELECT ?subject 
           (SAMPLE(?label) AS ?prefLabel) 
           (GROUP_CONCAT(DISTINCT ?p_id; separator=" | ") AS ?parentIDs) 
           (SAMPLE(?full_path) AS ?hierarchy)
           (SAMPLE(?description) AS ?scopeNote)
           (GROUP_CONCAT(DISTINCT ?altLabel; separator=" | ") AS ?synonyms) 
           (GROUP_CONCAT(DISTINCT ?crosswalk; separator=" | ") AS ?mappings)
    WHERE {{
      # Grab descendants OR the exact seed itself
      {{ ?subject gvp:broaderExtended aat:{seed_id} . }}
      UNION
      {{ BIND(aat:{seed_id} AS ?subject) }}

      ?subject skos:prefLabel ?label .
      FILTER(lang(?label) = "en")

      # The Single Preferred Path
      OPTIONAL {{ ?subject gvp:parentString ?full_path . }}
      
      # ALL Immediate Parents
      OPTIONAL {{ ?subject skos:broader ?parent . BIND(strafter(str(?parent), "aat/") AS ?p_id) }}
      
      # Synonyms & Scope Notes
      OPTIONAL {{ ?subject skos:altLabel ?altLabel . FILTER(lang(?altLabel) = "en") }}
      OPTIONAL {{ ?subject skos:scopeNote ?sn . ?sn dcterms:language gvp_lang:en ; rdf:value ?description . }}
      
      # Extract Crosswalks (exactMatch, closeMatch, sameAs)
      OPTIONAL {{
        {{ ?subject skos:exactMatch ?crosswalk . }}
        UNION
        {{ ?subject skos:closeMatch ?crosswalk . }}
        UNION
        {{ ?subject owl:sameAs ?crosswalk . }}
      }}
    }} 
    GROUP BY ?subject
    """

    try:
        response = requests.get(GETTY_SPARQL_URL, params={'query': QUERY, 'format': 'json'}, timeout=120)
        response.raise_for_status()
        results = response.json().get('results', {}).get('bindings', [])
        
        batch_rows = []
        for item in results:
            if 'subject' not in item or 'prefLabel' not in item: continue
                
            uri = item['subject']['value']
            if uri in visited_uris: continue
            
            cid = uri.split('/')[-1]
            label = item['prefLabel']['value']
            
            # --- Single Path Reversal Logic ---
            raw_path = item.get('hierarchy', {}).get('value', '')
            if raw_path:
                parts = [p.strip() for p in raw_path.split(',') if p.strip()]
                parts.reverse()
                hierarchy_path = " > ".join(parts) + f" > {label}"
            else:
                hierarchy_path = label

            extracted_data = {
                'Source_System': SOURCE_NAME,
                'Primary_Label': label,
                'CURIE': f"{SOURCE_PREFIX}:{cid}",
                'Formal_Label': label,
                'Concept_Type': 'skos:Concept',
                'Hierarchy_Path': hierarchy_path, 
                # Provide defaults if keys are missing from the SPARQL result
                'Synonyms': item.get('synonyms', {}).get('value', ''),
                'Description': item.get('scopeNote', {}).get('value', ''),
                'Parent_IDs': item.get('parentIDs', {}).get('value', ''), 
                'Concept_ID': cid,
                'URI': uri,
                'Has_Translation': "yes",
                'Status': 'active',
                'Crosswalks': item.get('mappings', {}).get('value', '')
            }
            
            batch_rows.append(finalize_row(extracted_data))
            visited_uris.add(uri)

        if batch_rows:
            batch_df = pd.DataFrame(batch_rows)[COLUMN_ORDER]
            batch_df.to_csv(
                raw_ingest_file, mode='a', index=False, 
                header=not os.path.isfile(raw_ingest_file), encoding='utf-8-sig'
            )
            print(f"    [+] Saved {len(batch_rows)} new unique concepts.")
        else:
            print(f"    [-] No new concepts found (or all were cached).")
            
        time.sleep(1) # Politeness delay between major queries

    except Exception as e:
        print(f"    [!] Error processing branch '{seed_name}': {e}")

print(f"\n[COMPLETE] Total unique Getty AAT concepts in file: {len(visited_uris)}")

In [ ]:
# ==============================================================================
# CELL 2: AAT LATERAL DISCOVERY (Tree-Reconstruction Format)
# Run this cell to harvest skos:related links outside the current scope.
# ==============================================================================

import os
import requests
import pandas as pd
import time
from dotenv import load_dotenv

RUN_LATERAL_DISCOVERY = False  # Toggle to False once your ingest script is finalized

if RUN_LATERAL_DISCOVERY:
    # --- 1. Setup & Baseline Load ---
    load_dotenv("../../config/.env")
    notebook_dir = os.path.dirname(os.path.abspath("__file__"))
    raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
    raw_aat_file = os.path.join(raw_data_dir, "raw_aat.csv")
    output_candidates_file = os.path.join(raw_data_dir, "aat_lateral_candidates.csv")
    
    GETTY_SPARQL_URL = "http://vocab.getty.edu/sparql.json"
    
    if not os.path.exists(raw_aat_file):
        raise FileNotFoundError("Please run Cell 1 to generate the baseline raw_aat.csv first.")
        
    existing_df = pd.read_csv(raw_aat_file)
    existing_uris = set(existing_df['URI'].astype(str).str.strip().str.rstrip('/'))
    print(f"[*] Loaded {len(existing_uris)} existing AAT concepts to serve as the baseline.")
    
    # --- 2. Chunking Logic for SPARQL ---
    uri_list = list(existing_uris)
    chunk_size = 100 
    chunks = [uri_list[i:i + chunk_size] for i in range(0, len(uri_list), chunk_size)]
    
    candidates_dict = {} # Deduplication dictionary
    
    print(f"\nScanning Getty AAT via SPARQL for lateral associations across {len(chunks)} batches...\n")
    
    for idx, chunk in enumerate(chunks, 1):
        print(f"\rProcessing batch {idx}/{len(chunks)}...", end="", flush=True)
        
        values_str = " ".join([f"<{uri}>" for uri in chunk if uri.startswith("http")])
        if not values_str: continue

        QUERY = f"""
        SELECT DISTINCT ?candidate ?candidateLabel ?full_path ?parent ?parentLabel
        WHERE {{
          VALUES ?existing {{ {values_str} }}
          
          ?existing skos:related ?candidate .
          ?candidate skos:prefLabel ?candidateLabel .
          FILTER(lang(?candidateLabel) = "en")
          
          OPTIONAL {{ ?candidate gvp:parentString ?full_path . }}
          
          OPTIONAL {{
             ?candidate gvp:broaderPreferred ?parent .
             ?parent skos:prefLabel ?parentLabel .
             FILTER(lang(?parentLabel) = "en")
          }}
        }}
        """
        
        try:
            res = requests.get(GETTY_SPARQL_URL, params={'query': QUERY, 'format': 'json'}, timeout=60)
            if res.status_code != 200:
                time.sleep(2)
                continue
                
            results = res.json().get('results', {}).get('bindings', [])
            
            for item in results:
                c_uri = item.get('candidate', {}).get('value', '').rstrip('/')
                c_label = item.get('candidateLabel', {}).get('value', 'Unknown')
                
                parent_uri = item.get('parent', {}).get('value', '').rstrip('/')
                parent_label = item.get('parentLabel', {}).get('value', 'Unknown Parent')
                
                # --- Path Construction ---
                raw_path = item.get('full_path', {}).get('value', '')
                parts = [p.strip() for p in raw_path.split(',') if p.strip()]
                parts.reverse()
                
                # The parent's path is everything EXCEPT the candidate
                parent_path = " > ".join(parts) if parts else parent_label
                # The candidate's path appends the candidate to the parent path
                candidate_path = f"{parent_path} > {c_label}" if parts else c_label
                
                # --- 1. Add Candidate Row ---
                if c_uri and c_uri not in existing_uris:
                    candidates_dict[c_uri] = {
                        'Candidate_URI': c_uri,
                        'Candidate_Label': c_label,
                        'Hierarchy_Path': candidate_path
                    }
                    
                # --- 2. Add Parent Row ---
                if parent_uri and parent_uri not in existing_uris:
                    if parent_uri not in candidates_dict:
                        candidates_dict[parent_uri] = {
                            'Candidate_URI': parent_uri,
                            'Candidate_Label': parent_label, # Tag removed here
                            'Hierarchy_Path': parent_path
                        }
                    
        except Exception:
            pass 
            
        time.sleep(0.5)

    # --- 3. Export and Sort ---
    print(f"\n\nLateral Discovery Complete! Found {len(candidates_dict)} unique candidates and parents.")
    
    if candidates_dict:
        candidates_df = pd.DataFrame(list(candidates_dict.values()))
        
        # Sort by hierarchy so parents sit immediately above their children
        candidates_df = candidates_df.sort_values(by='Hierarchy_Path')
        candidates_df.to_csv(output_candidates_file, index=False, encoding='utf-8-sig')
        print(f"Sorted candidates saved to: {output_candidates_file}")
        print("\nNext Step: Review the CSV and copy the URIs of the relevant root branches into Cell 1!")
    else:
        print("No new relevant concepts found. Your seed list is comprehensive!")
else:
    print("Lateral Discovery is disabled.")

In [ ]:
import requests
import json

AAT_SEEDS = {
    # --- Original High-Level Seeds ---
    "religious organizations": "300250844",
    "people in religion and related occupations": "300025746",
    "religions and religious concepts": "300055980",
    "religious structures": "300120364",
    "religious centers": "300387272",
    "religious events": "300069688",
    "religious institutions": "300343472",

    # --- New Seeds from Lateral Discovery ---
    "<religion and related disciplines>": "300054299",
    "occult sciences": "300054587",
    "christenings": "300137692",
    "religious holidays": "300069349",
    "religious seasons": "300192156",
    "rituals (events)": "300065284",
    "pilgrimages": "300069279",
    "conclaves": "300404871",
    "yoga (activity)": "300264366",
    "ritual destruction (process)": "300458782",
    "religious divisions": "300236153",
    "papacies": "300379037",
    "religious characters": "300412107",
    "retablistas": "300311286",
    "<people in the occult sciences>": "300239172",
    "buddhas (people)": "300404698",
    "pilgrims (people)": "300266259",
    "saints": "300150555",
    "teixiptlahuan": "300404940",
    "cults (group or movement)": "300251847",
    "beati (people)": "300150482",
    "gnostics": "300236781",
    "holy water": "300264702",
    "chrism": "300265211",
    "peyote (plant material)": "300249790",
    "missions (complexes)": "300410262",
    "religious complexes": "300263741",
    "sacred sites": "300386998",
    "archiepiscopal sees": "300387236",
    "episcopal sees": "300387235",
    "agrahāra": "300386139",
    "altar tombs": "300005928",
    "sacred tanks": "300458692",
    "cross monuments": "300133797",
    "clergy houses": "300005639",
    "abbot's quarters": "300451314",
    "ceremonial containers": "300197576",
    "christening cups": "300391137",
    "liturgical costume": "300210414",
    "religious habits": "300390957",
    "personal shrines": "300391181",
    "totems": "300255468",
    "shamanic art": "300457672",

    # --- Additional Seeds round 2 ---
    "rites": "300312160",
    "fasts (events)": "300069095",
    "ritual rebuilding (process)": "300458781",
    "convents (groups)": "300312089",
    "religious schools (institutions)": "300343470",
    "theological seminaries (institutions)": "300343434",
    "seminarians": "300404369",
    "altar makers": "300420020",
    "altarpiece artists": "300435119",
    "religious poetry": "300417816",
    "prayers (literary genre)": "300026452",
    "religious literature (genre)": "300417817",
    "alms (gifts)": "300400538",
    "religious schools (buildings)": "300448956",
    "prayer stools (stools, general)": "300038454",
    "baptismal candles": "300404224",
    "epiphany candles": "300404229",
    "paschal candles": "300190806",
    "epiphany candlesticks": "300404246",
    "religious objects": "300234098",
    "religious calendars": "300026754",
    "Torah scrolls": "300419304",
    "sermons": "300026669",
    "totem poles": "300184947",

    # --- Additional seeds round 3 ---
    "parochial schools (buildings)": "300006580",
    "Sunday schools (buildings)": "300132249",
    "madrasas (buildings)": "300006563",
    "yeshivas (buildings)": "300048798",
    "sacred objects": "300234190",
    "prayers (compositions)": "300400521"
}

GETTY_SPARQL_URL = "http://vocab.getty.edu/sparql.json"
redundant_seeds = []

print("Analyzing seeds for hierarchical redundancy...")

# We ask Getty: For all our seeds, do any of them have a broader parent that is ALSO in our seed list?
values_str = " ".join([f"aat:{seed_id}" for seed_id in AAT_SEEDS.values()])

QUERY = f"""
SELECT DISTINCT ?child ?parent
WHERE {{
  VALUES ?child {{ {values_str} }}
  VALUES ?parent {{ {values_str} }}
  
  ?child gvp:broaderExtended ?parent .
  FILTER(?child != ?parent)
}}
"""

res = requests.get(GETTY_SPARQL_URL, params={'query': QUERY, 'format': 'json'})
results = res.json().get('results', {}).get('bindings', [])

for item in results:
    child_id = item['child']['value'].split('/')[-1]
    parent_id = item['parent']['value'].split('/')[-1]
    
    # Find the names for printing
    child_name = next(name for name, i in AAT_SEEDS.items() if i == child_id)
    parent_name = next(name for name, i in AAT_SEEDS.items() if i == parent_id)
    
    print(f"[REDUNDANT] '{child_name}' can be removed. It is a descendant of '{parent_name}'.")
    redundant_seeds.append(child_name)

print(f"\nTotal redundant seeds found: {len(set(redundant_seeds))}")